In [7]:
train_sets = ['../data/raw/sample_23.csv']
test_set = '../data/raw/sample_24.csv'
max_rows = 100

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Data loading
print("Data loading...")
train_path = train_sets[0] if isinstance(train_sets, (list, tuple)) else train_sets
test_path = test_set

train_wide = pd.read_csv(train_path, nrows=max_rows)
test_wide = pd.read_csv(test_path, nrows=max_rows)

id_col = "ID"
if id_col not in train_wide.columns:
    id_col = train_wide.columns[0]

train_date_cols = [c for c in train_wide.columns if c != id_col and pd.notna(pd.to_datetime(c, errors="coerce"))]
test_date_cols = [c for c in test_wide.columns if c != id_col and pd.notna(pd.to_datetime(c, errors="coerce"))]

if len(train_date_cols) == 0 or len(test_date_cols) == 0:
    raise ValueError("No valid date columns were detected. Expected columns like YYYY-MM-DD.")

# Data conversion to long columns instead of wide rows
print("Converting data")
train_long = train_wide.melt(id_vars=id_col, value_vars=train_date_cols, var_name="date", value_name="y")
train_long["date"] = pd.to_datetime(train_long["date"])
train_long = train_long.sort_values([id_col, "date"]).reset_index(drop=True)

test_long_actual = test_wide.melt(id_vars=id_col, value_vars=test_date_cols, var_name="date", value_name="y_true")
test_long_actual["date"] = pd.to_datetime(test_long_actual["date"])

# Helper functions
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["dow"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["day"] = out["date"].dt.day
    out["dayofyear"] = out["date"].dt.dayofyear

    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7.0)
    return out


def add_lag_features(df: pd.DataFrame, group_col: str, target_col: str) -> pd.DataFrame:
    out = df.copy()
    g = out.groupby(group_col)[target_col]

    out["lag_1"] = g.shift(1)
    out["lag_7"] = g.shift(7)
    out["lag_14"] = g.shift(14)
    out["lag_28"] = g.shift(28)

    out["roll_mean_7"] = g.shift(1).rolling(7).mean().reset_index(level=0, drop=True)
    out["roll_mean_28"] = g.shift(1).rolling(28).mean().reset_index(level=0, drop=True)
    return out

print("Adding features")
train_feat = add_calendar_features(train_long)
train_feat = add_lag_features(train_feat, group_col=id_col, target_col="y")

feature_cols = [
    "dow", "month", "day", "dayofyear",
    "doy_sin", "doy_cos", "dow_sin", "dow_cos",
    "lag_1", "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_mean_28",
]

# Remove zero values
train_model_df = train_feat.dropna(subset=feature_cols + ["y"]).copy()
X_train = train_model_df[feature_cols]
y_train = train_model_df["y"]

# Model
print("Fitting model")
model = HistGradientBoostingRegressor(
    max_depth=8,
    learning_rate=0.05,
    max_iter=300,
    random_state=42,
)
model.fit(X_train, y_train)

# Test data loading
print("Loading test data")
future_dates = pd.to_datetime(test_date_cols)
future_dates = np.array(sorted(future_dates))

id_values = sorted(set(train_wide[id_col]).intersection(set(test_wide[id_col])))
if len(id_values) == 0:
    raise ValueError("No overlapping IDs between train and test files.")

train_indexed = train_long.set_index([id_col, "date"])["y"].sort_index()
pred_rows = []

# Predicting
print("Predicting...")
for item_id in id_values:
    # loading historical data
    hist_series = train_indexed.loc[item_id].sort_index()
    history = hist_series.tolist()

    for dt in future_dates:
        # determining features for the current prediction date in history
        lag_1 = history[-1] if len(history) >= 1 else np.nan
        lag_7 = history[-7] if len(history) >= 7 else np.nan
        lag_14 = history[-14] if len(history) >= 14 else np.nan
        lag_28 = history[-28] if len(history) >= 28 else np.nan

        roll_mean_7 = np.mean(history[-7:]) if len(history) >= 7 else np.nan
        roll_mean_28 = np.mean(history[-28:]) if len(history) >= 28 else np.nan

        dow = dt.dayofweek
        month = dt.month
        day = dt.day
        dayofyear = dt.dayofyear

        row = {
            "dow": dow,
            "month": month,
            "day": day,
            "dayofyear": dayofyear,
            "doy_sin": np.sin(2 * np.pi * dayofyear / 366.0),
            "doy_cos": np.cos(2 * np.pi * dayofyear / 366.0),
            "dow_sin": np.sin(2 * np.pi * dow / 7.0),
            "dow_cos": np.cos(2 * np.pi * dow / 7.0),
            "lag_1": lag_1,
            "lag_7": lag_7,
            "lag_14": lag_14,
            "lag_28": lag_28,
            "roll_mean_7": roll_mean_7,
            "roll_mean_28": roll_mean_28,
        }

        x_row = pd.DataFrame([row], columns=feature_cols)

        for col in ["lag_7", "lag_14", "lag_28", "roll_mean_7", "roll_mean_28"]:
            if pd.isna(x_row.at[0, col]):
                x_row.at[0, col] = x_row.at[0, "lag_1"]

        # predict test datapoint
        y_pred = model.predict(x_row)[0]
        pred_rows.append({id_col: item_id, "date": dt, "y_pred": y_pred})

        history.append(y_pred)

pred_long = pd.DataFrame(pred_rows)

# Verify predicted values
print("Verifying predicted values")
eval_df = pred_long.merge(test_long_actual, on=[id_col, "date"], how="left")
if eval_df["y_true"].notna().any():
    mae = mean_absolute_error(eval_df["y_true"], eval_df["y_pred"])
    rmse = np.sqrt(mean_squared_error(eval_df["y_true"], eval_df["y_pred"]))
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
else:
    print("No non-null y_true values found in test file; skipped evaluation.")

pred_wide = pred_long.copy()
pred_wide["date"] = pred_wide["date"].dt.strftime("%Y-%m-%d")
pred_wide = pred_wide.pivot(index=id_col, columns="date", values="y_pred").reset_index()

pred_out_path = "../data/processed/sklearn_forecast_2024.csv"
pred_wide.to_csv(pred_out_path, index=False)

print(f"Saved predictions to: {pred_out_path}")
pred_wide.head()

Data loading...
Converting data
Adding features
Fitting model
Loading test data
MAE:  4.2194
RMSE: 6.6645
Saved predictions to: ../data/processed/sklearn_forecast_2024.csv


date,ID,2024-01-01,2024-01-02,2024-01-03,2024-01-04,2024-01-05,2024-01-06,2024-01-07,2024-01-08,2024-01-09,...,2024-12-22,2024-12-23,2024-12-24,2024-12-25,2024-12-26,2024-12-27,2024-12-28,2024-12-29,2024-12-30,2024-12-31
0,22,10.376967,10.150577,10.273509,10.442461,10.049628,10.953093,11.021276,10.667622,10.694864,...,11.049520,10.451104,10.444039,10.444039,10.436520,10.387885,10.883211,11.007081,10.255546,10.248481
1,42,40.664148,36.800485,36.660440,37.389506,36.616644,37.270643,39.116479,36.892503,33.991187,...,20.481738,18.944028,19.539393,19.330003,19.345666,19.288829,19.799251,19.950746,18.742790,18.875829
2,56,7.357146,7.459440,6.820784,8.138848,8.079860,9.013588,9.005913,8.712747,8.929460,...,10.640239,9.949394,9.929385,9.929385,9.921865,10.002052,10.655699,10.640239,9.924352,9.904343
3,58,15.692685,16.165056,15.986810,14.703202,14.896211,16.132909,15.425408,14.331069,14.587092,...,13.934095,13.292952,13.523008,13.538189,13.530670,13.560617,13.965300,13.850718,13.391345,13.497967
4,64,2.515100,2.515100,2.515100,2.580786,2.580786,2.674451,2.662215,2.641912,2.641912,...,2.683592,2.663289,2.663289,2.663289,2.663289,2.663289,2.715173,2.683592,2.663289,2.663289
